# 49 — Inpainting Analysis (Spatial Reconstruction at $t=0$)

How well does v27 reconstruct masked stations at the current time ($\Delta=0$)?
We examine MAE as a function of neighbour density, elevation, terrain class,
and nearest-neighbour distance, then compare masked-vs-visible convergence
over lead time.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats

import common

# Force a clean, explicit style -- this kernel is shared across notebooks in
# the same session, and an earlier one may have left a dark style (e.g.
# plt.style.use('dark_background')) in matplotlib's global rcParams. A
# partial rcParams.update() here would leave axes.facecolor/text colors from
# that style in place, producing black axes with invisible ticks/labels.
plt.style.use('default')
mpl.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'savefig.facecolor': 'white', 'savefig.bbox': 'tight',
    'axes.edgecolor': '0.3', 'axes.labelcolor': 'black',
    'xtick.color': '0.3', 'ytick.color': '0.3', 'text.color': 'black',
})

FIG_DIR = common.FIG
os.makedirs(FIG_DIR, exist_ok=True)
print('OK')

In [ ]:
acc = common.load_agg('v27', 'mr0.50')
stn = common.station_table()
grid = acc['grid']  # delta steps

# Identify t=0 index
k0 = int(np.where(grid == 0)[0][0])
print(f'Lead-time grid: {grid}')
print(f't=0 index: k0 = {k0}')
print(f'Stations: {len(stn)}, Variables: {acc["mod_msk_cnt"].shape[-1]}')

VARS = common.norm_stats()['var_names']
UNITS = [common.UNITS.get(v, '') for v in VARS]
print(f'Variables: {VARS}')

In [ ]:
# Overview statistics at t=0
mae_msk_0 = common.metric(acc, 'mae', 'mod', 'msk')  # shape (K, N, V)
mae_vis_0 = common.metric(acc, 'mae', 'mod', 'vis')   # shape (K, N, V)

# Station-averaged at t=0
msk_t0 = mae_msk_0[k0]  # (N, V)
vis_t0 = mae_vis_0[k0]  # (N, V)

header = '{:>12s}'.format('Variable')
for tag in ['masked', 'visible', 'ratio']:
    header += '  {:>10s}'.format(tag)
print(header)
print('-' * len(header))

for v, vname in enumerate(VARS):
    m_mean = np.nanmean(msk_t0[:, v])
    v_mean = np.nanmean(vis_t0[:, v])
    ratio = m_mean / v_mean if v_mean > 0 else float('nan')
    row = '{:>12s}'.format(vname)
    row += '  {:10.4f}'.format(m_mean)
    row += '  {:10.4f}'.format(v_mean)
    row += '  {:10.2f}'.format(ratio)
    print(row)


In [ ]:
# Fig 1: Distribution of MAE at t=0 for masked vs visible stations
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
fig.suptitle(r'MAE at $\Delta=0$: masked vs visible stations', fontsize=13)

for v, (ax, vname) in enumerate(zip(axes.flat, VARS)):
    m_vals = msk_t0[:, v]
    v_vals = vis_t0[:, v]
    m_vals = m_vals[np.isfinite(m_vals)]
    v_vals = v_vals[np.isfinite(v_vals)]
    bins = np.linspace(0, np.percentile(np.concatenate([m_vals, v_vals]), 97), 40)
    ax.hist(v_vals, bins, alpha=0.55, label='visible', color='#5BA55B')
    ax.hist(m_vals, bins, alpha=0.55, label='masked',  color='#D35F5F')
    ax.set_title(vname, fontsize=10)
    ax.set_xlabel(f'MAE [{UNITS[v]}]', fontsize=8)
    ax.set_ylabel('stations', fontsize=8)
    ax.legend(fontsize=7)

fig.savefig(os.path.join(FIG_DIR, '49_msk_vs_vis_t0_hist.png'), dpi=200)
plt.show()
plt.close(fig)
print('Fig 1 saved')

In [ ]:
# Fig 2: MAE(t=0, masked) vs neighbour counts at 5/10/25/50 km
radii = ['n_within_5km', 'n_within_10km', 'n_within_25km', 'n_within_50km']
radius_labels = ['5 km', '10 km', '25 km', '50 km']

fig, axes = plt.subplots(len(VARS), len(radii), figsize=(16, 14),
                         constrained_layout=True, sharex='col')
fig.suptitle(r'Inpainting MAE at $\Delta=0$ vs neighbour count (masked stations)',
             fontsize=13)

for v, vname in enumerate(VARS):
    y = msk_t0[:, v]
    mask = np.isfinite(y)
    for j, (rcol, rlbl) in enumerate(zip(radii, radius_labels)):
        ax = axes[v, j]
        x = stn[rcol].values
        ax.scatter(x[mask], y[mask], s=8, alpha=0.5, color='#D35F5F', edgecolor='none')
        if mask.sum() > 5:
            slope, intercept, r, p, _ = stats.linregress(x[mask], y[mask])
            xfit = np.linspace(x[mask].min(), x[mask].max(), 50)
            ax.plot(xfit, slope*xfit+intercept, 'k--', lw=0.8, alpha=0.6)
            ax.text(0.95, 0.95, f'r={r:.2f}', transform=ax.transAxes,
                    fontsize=7, ha='right', va='top')
        if v == 0:
            ax.set_title(rlbl, fontsize=9)
        if j == 0:
            ax.set_ylabel(f'{vname}\nMAE [{UNITS[v]}]', fontsize=7)
        if v == len(VARS)-1:
            ax.set_xlabel('# neighbours', fontsize=8)

fig.savefig(os.path.join(FIG_DIR, '49_mae_vs_neighbours_grid.png'), dpi=200)
plt.show()
plt.close(fig)
print('Fig 2 saved')

In [ ]:
# Fig 3: Main-paper candidate — MAE vs n_within_25km (one panel per variable)
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
fig.suptitle(r'Inpainting MAE at $\Delta=0$ vs neighbours within 25 km\n(masked stations)',
             fontsize=13)

for v, (ax, vname) in enumerate(zip(axes.flat, VARS)):
    y = msk_t0[:, v]
    x = stn['n_within_25km'].values
    mask = np.isfinite(y)
    ax.scatter(x[mask], y[mask], s=18, alpha=0.5, color='#2C7A5E', edgecolor='none')
    if mask.sum() > 5:
        slope, intercept, r, p, _ = stats.linregress(x[mask], y[mask])
        xfit = np.linspace(x[mask].min(), x[mask].max(), 50)
        ax.plot(xfit, slope*xfit+intercept, 'k--', lw=1)
        pstr = f'p={p:.1e}' if p < 0.001 else f'p={p:.3f}'
        ax.text(0.95, 0.95, f'r={r:.2f}, {pstr}',
                transform=ax.transAxes, fontsize=7, ha='right', va='top')
    ax.set_title(f'{vname} [{UNITS[v]}]', fontsize=10)
    ax.set_xlabel('# neighbours within 25 km', fontsize=8)
    ax.set_ylabel('MAE', fontsize=8)

fig.savefig(os.path.join(FIG_DIR, '49_mae_vs_n25km.png'), dpi=200)
plt.show()
plt.close(fig)
print('Fig 3 saved')

In [ ]:
# Fig 4: Inpainting MAE vs station elevation
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
fig.suptitle(r'Inpainting MAE at $\Delta=0$ vs elevation (masked stations)',
             fontsize=13)

for v, (ax, vname) in enumerate(zip(axes.flat, VARS)):
    y = msk_t0[:, v]
    x = stn['height'].values
    mask = np.isfinite(y)
    ax.scatter(x[mask], y[mask], s=18, alpha=0.5, color='#7B68AE', edgecolor='none')
    if mask.sum() > 5:
        slope, intercept, r, p, _ = stats.linregress(x[mask], y[mask])
        xfit = np.linspace(x[mask].min(), x[mask].max(), 50)
        ax.plot(xfit, slope*xfit+intercept, 'k--', lw=1)
        ax.text(0.95, 0.95, f'r={r:.2f}', transform=ax.transAxes,
                fontsize=7, ha='right', va='top')
    ax.set_title(f'{vname} [{UNITS[v]}]', fontsize=10)
    ax.set_xlabel('Elevation [m]', fontsize=8)
    ax.set_ylabel('MAE', fontsize=8)

fig.savefig(os.path.join(FIG_DIR, '49_mae_vs_elevation.png'), dpi=200)
plt.show()
plt.close(fig)
print('Fig 4 saved')

In [ ]:
# Fig 5: Terrain class boxplots at t=0
tc_idx = common.terrain_class_indices(stn)
tc_names = list(tc_idx.keys())

fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
fig.suptitle(r'Inpainting MAE at $\Delta=0$ by terrain class (masked stations)',
             fontsize=13)

colors = ['#5BA55B', '#D35F5F', '#7BAFD4', '#E8A838']

for v, (ax, vname) in enumerate(zip(axes.flat, VARS)):
    data = []
    labels = []
    for tc_name in tc_names:
        idx = tc_idx[tc_name]
        vals = msk_t0[idx, v]
        vals = vals[np.isfinite(vals)]
        data.append(vals)
        labels.append(tc_name)
    bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, showfliers=False)
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.6)
    ax.set_title(f'{vname} [{UNITS[v]}]', fontsize=10)
    ax.tick_params(axis='x', rotation=25, labelsize=7)
    ax.set_ylabel('MAE', fontsize=8)

fig.savefig(os.path.join(FIG_DIR, '49_terrain_class_t0.png'), dpi=200)
plt.show()
plt.close(fig)
print('Fig 5 saved')

In [ ]:
# Fig 6: Masked and visible MAE convergence over lead time
mae_msk_all = common.metric(acc, 'mae', 'mod', 'msk', pool=('N', 'V'))  # (K,)
mae_vis_all = common.metric(acc, 'mae', 'mod', 'vis', pool=('N', 'V'))  # (K,)
lead_hours = grid * 10 / 60  # convert steps to hours

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

ax1.plot(lead_hours, mae_msk_all, 'o-', color='#D35F5F', label='masked', ms=5)
ax1.plot(lead_hours, mae_vis_all, 's-', color='#5BA55B', label='visible', ms=5)
ax1.set_xlabel('Lead time [h]', fontsize=10)
ax1.set_ylabel('MAE (all variables, physical units)', fontsize=10)
ax1.set_title('Absolute MAE by lead time', fontsize=11)
ax1.legend(fontsize=9)
ax1.axvline(0.5, color='gray', ls=':', lw=0.8, alpha=0.6)
ax1.text(0.55, ax1.get_ylim()[1]*0.95, r'$\Delta$=30 min', fontsize=7, color='gray')

ratio = mae_msk_all / mae_vis_all
ax2.plot(lead_hours, ratio, 'D-', color='#7B68AE', ms=5)
ax2.axhline(1.0, color='gray', ls='--', lw=0.8)
ax2.set_xlabel('Lead time [h]', fontsize=10)
ax2.set_ylabel('MAE ratio (masked / visible)', fontsize=10)
ax2.set_title('Convergence of masked toward visible', fontsize=11)
ax2.axvline(0.5, color='gray', ls=':', lw=0.8, alpha=0.6)

fig.savefig(os.path.join(FIG_DIR, '49_msk_vis_convergence.png'), dpi=200)
plt.show()
plt.close(fig)
print('Fig 6 saved')

In [ ]:
# Fig 7: Inpainting MAE vs nearest-neighbour distance
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
fig.suptitle(r'Inpainting MAE at $\Delta=0$ vs nearest-neighbour distance\n(masked stations)',
             fontsize=13)

for v, (ax, vname) in enumerate(zip(axes.flat, VARS)):
    y = msk_t0[:, v]
    x = stn['nn_dist_km'].values
    mask = np.isfinite(y)
    ax.scatter(x[mask], y[mask], s=18, alpha=0.5, color='#D4A76A', edgecolor='none')
    if mask.sum() > 5:
        slope, intercept, r, p, _ = stats.linregress(x[mask], y[mask])
        xfit = np.linspace(x[mask].min(), x[mask].max(), 50)
        ax.plot(xfit, slope*xfit+intercept, 'k--', lw=1)
        pstr = f'p={p:.1e}' if p < 0.001 else f'p={p:.3f}'
        ax.text(0.95, 0.95, f'r={r:.2f}, {pstr}',
                transform=ax.transAxes, fontsize=7, ha='right', va='top')
    ax.set_title(f'{vname} [{UNITS[v]}]', fontsize=10)
    ax.set_xlabel('Nearest-neighbour distance [km]', fontsize=8)
    ax.set_ylabel('MAE', fontsize=8)

fig.savefig(os.path.join(FIG_DIR, '49_mae_vs_nn_dist.png'), dpi=200)
plt.show()
plt.close(fig)
print('Fig 7 saved')

## Fixed-mask station analysis

Reconstruct the fixed evaluation mask (seed=42, MR=0.5) to identify
which 77 stations are masked and which 78 are visible, then compute
spatial metrics relative to the *visible* set: nearest visible neighbour
distance, number of visible neighbours within 10/25/50 km, and elevation
difference to the nearest visible station.

In [ ]:
import torch
from scipy.spatial.distance import cdist

# ── Mask partition ─────────────────────────────────────────────────────────
# NOTE (2026-09-01): the evaluation mask is NOT fixed. predictions.pt holds
# 11,684 distinct masks (one per window, 77 stations each); every station is
# masked in ~50% of windows. Reconstructing a single mask from
# torch.manual_seed(42) does NOT reproduce the dump. The analysis below uses
# the FIRST window's partition as an illustrative split; the per-station
# masked-window MAE (msk_t0) is valid for all 155 stations regardless.
N = len(stn)
_d5 = common.load_dump('v27', 'mr0.50')
MASKED_IDX = np.sort(_d5['masked_idx'][0].numpy())
VISIBLE_IDX = np.setdiff1d(np.arange(N), MASKED_IDX)
del _d5
num_masked = len(MASKED_IDX)
print(f'First-window mask: {len(MASKED_IDX)} masked, {len(VISIBLE_IDX)} visible')

is_masked = np.zeros(N, dtype=bool)
is_masked[MASKED_IDX] = True

# ── Coordinates in CH1903+ (easting/northing in metres) ──────────────────
coords = np.column_stack([stn.easting.values, stn.northing.values])
vis_coords = coords[VISIBLE_IDX]
msk_coords = coords[MASKED_IDX]

# ── Pairwise distances masked→visible (km) ──────────────────────────────
D_m2v = cdist(msk_coords, vis_coords) / 1000.0  # (n_msk, n_vis) in km

# Per masked station: nearest visible distance, visible counts within radii
nn_vis_dist_km = D_m2v.min(axis=1)  # (n_msk,)
nn_vis_idx = D_m2v.argmin(axis=1)  # index into VISIBLE_IDX

n_vis_10km = (D_m2v < 10).sum(axis=1)
n_vis_25km = (D_m2v < 25).sum(axis=1)
n_vis_50km = (D_m2v < 50).sum(axis=1)

# Elevation difference to nearest visible station
h_masked = stn.height.values[MASKED_IDX]
h_vis_nn = stn.height.values[VISIBLE_IDX[nn_vis_idx]]
elev_diff_nn = np.abs(h_masked - h_vis_nn)

# Terrain class for masked stations
tc_idx_msk = common.terrain_class_indices(stn)

print(f'Nearest visible distance: {nn_vis_dist_km.min():.1f}–{nn_vis_dist_km.max():.1f} km')
print(f'Visible within 10 km: {n_vis_10km.min()}–{n_vis_10km.max()}')
print(f'Visible within 25 km: {n_vis_25km.min()}–{n_vis_25km.max()}')
print(f'|Δh| to nearest visible: {elev_diff_nn.min():.0f}–{elev_diff_nn.max():.0f} m')


### Map: T=0 reconstruction MAE for masked stations

Masked stations coloured by MAE; visible stations shown as small grey dots.

In [ ]:
# ── DEM + Swiss border ──────────────────────────────────────────────────────
import geopandas as gpd
import matplotlib.colors as mcolors
try:
    import rioxarray  # noqa
except ImportError:
    pass

PROJ = os.path.abspath(os.path.join(os.getcwd(), '..', '..')) \
       if os.path.isfile('common.py') else os.getcwd()
if os.path.join(PROJ, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJ, 'src'))

PATH_SWISSSHAPE = os.path.expanduser(
    os.environ.get('SWISSSHAPE',
        os.path.join(PROJ, 'swissboundaries3d_2056.shp.zip')))
_CAND = [os.environ.get('DATA_ROOT', ''),
         os.path.expanduser('~/PeakWeatherDataset'),
         os.path.join(PROJ, 'PeakWeatherDataset')]
DATA_ROOT = next((p for p in _CAND if os.path.isdir(str(p))), _CAND[-1])

from peakweather.dataset import PeakWeatherDataset
ds_topo = PeakWeatherDataset(
    root=DATA_ROOT,
    parameters=['temperature', 'pressure', 'humidity',
                 'wind_speed', 'wind_direction', 'precipitation'],
    compute_uv=True, station_type='meteo_station',
    imputation_method=None, freq='d', extended_topo_vars='DEM')

def _load_dem_and_border(ds_topo, path_swissshape, coarsen=10):
    switzerland = gpd.read_file(
        path_swissshape,
        layer='swissBOUNDARIES3D_1_5_TLM_LANDESGEBIET').to_crs('EPSG:2056')
    topo = ds_topo.load_topography()
    dem  = topo['topo_DEM'].dem
    dem_ch = dem.rio.clip(switzerland.geometry, switzerland.crs, drop=False)
    minx, miny, maxx, maxy = switzerland.total_bounds
    dem_bg = dem.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()
    dem_fg = dem_ch.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()
    dem_bg = dem_bg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    dem_fg = dem_fg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    return dem_bg, dem_fg, switzerland

def draw_dem(ax, dem_bg, dem_fg, switzerland):
    norm = mcolors.Normalize(vmin=0, vmax=4500)
    dem_bg.plot(ax=ax, cmap='terrain', norm=norm, alpha=0.35,
                robust=True, add_labels=False, add_colorbar=False)
    dem_fg.plot(ax=ax, cmap='terrain', norm=norm,
                robust=True, add_labels=False, add_colorbar=False)
    switzerland.boundary.plot(ax=ax, color='white', linewidth=1.0)
    ax.axis('off')

print('Loading DEM + border ...')
dem_bg, dem_fg, switzerland = _load_dem_and_border(ds_topo, PATH_SWISSSHAPE)
print('Done.')


In [ ]:
# ── Map: T=0 reconstruction MAE per masked station ─────────────────────────
NV = len(VARS)
fig, axes = plt.subplots(1, NV, figsize=(6.0 * NV, 7.5))
for vi, (ax, vname) in enumerate(zip(axes, VARS)):
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    # Visible stations as small grey dots
    ax.scatter(stn.easting.values[VISIBLE_IDX], stn.northing.values[VISIBLE_IDX],
               s=10, c='0.6', alpha=0.4, zorder=4)
    # Masked stations coloured by MAE
    mae_vals = msk_t0[MASKED_IDX, vi]
    valid = np.isfinite(mae_vals)
    sc = ax.scatter(stn.easting.values[MASKED_IDX[valid]],
                    stn.northing.values[MASKED_IDX[valid]],
                    c=mae_vals[valid], s=55, cmap='YlOrRd',
                    edgecolors='k', linewidths=0.3, zorder=5)
    fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02,
                 label=f'MAE [{UNITS[vi]}]')
    ax.set_title(f'{vname} at Δ=0', fontsize=12)

fig.suptitle('T=0 reconstruction MAE for masked stations (MAE Transformer, MR=0.5)',
             y=1.02)
plt.tight_layout()
common.save_fig(fig, '49_t0_mae_map')
plt.show()
plt.close(fig)


### Reconstruction MAE vs visible-neighbour spatial metrics

Scatter plots of per-station masked MAE at T=0 against metrics computed
relative to the *visible* station set: distance to nearest visible station,
number of visible stations within 10/25/50 km, and absolute elevation
difference to the nearest visible station.

In [ ]:
# ── MAE vs nearest visible station distance ─────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
fig.suptitle('Reconstruction MAE at Δ=0 vs nearest visible station distance\n'
             '(masked stations, v27 MR=0.5)', fontsize=13)

for v, (ax, vname) in enumerate(zip(axes.flat, VARS)):
    y = msk_t0[MASKED_IDX, v]
    x = nn_vis_dist_km
    mask = np.isfinite(y)
    ax.scatter(x[mask], y[mask], s=18, alpha=0.5, color='#D4A76A', edgecolor='none')
    if mask.sum() > 5:
        slope, intercept, r, p, _ = stats.linregress(x[mask], y[mask])
        xfit = np.linspace(x[mask].min(), x[mask].max(), 50)
        ax.plot(xfit, slope*xfit+intercept, 'k--', lw=1)
        pstr = f'p={p:.1e}' if p < 0.001 else f'p={p:.3f}'
        ax.text(0.95, 0.95, f'r={r:.2f}, {pstr}\nn={mask.sum()}',
                transform=ax.transAxes, fontsize=7, ha='right', va='top')
    ax.set_title(f'{vname} [{UNITS[v]}]', fontsize=10)
    ax.set_xlabel('Distance to nearest visible station [km]', fontsize=8)
    ax.set_ylabel('MAE', fontsize=8)

common.save_fig(fig, '49_mae_vs_nn_vis_dist')
plt.show()
plt.close(fig)


In [ ]:
# ── MAE vs visible neighbours within 10/25/50 km ────────────────────────────
vis_radii = [('10 km', n_vis_10km), ('25 km', n_vis_25km), ('50 km', n_vis_50km)]

fig, axes = plt.subplots(len(VARS), len(vis_radii), figsize=(14, 14),
                         constrained_layout=True, sharex='col')
fig.suptitle('Reconstruction MAE at Δ=0 vs visible neighbours within radius\n'
             '(masked stations, v27 MR=0.5)', fontsize=13)

for v, vname in enumerate(VARS):
    y = msk_t0[MASKED_IDX, v]
    mask = np.isfinite(y)
    for j, (rlbl, x) in enumerate(vis_radii):
        ax = axes[v, j]
        ax.scatter(x[mask], y[mask], s=12, alpha=0.5, color='#D35F5F', edgecolor='none')
        if mask.sum() > 5:
            slope, intercept, r, p, _ = stats.linregress(x[mask], y[mask])
            xfit = np.linspace(x[mask].min(), x[mask].max(), 50)
            ax.plot(xfit, slope*xfit+intercept, 'k--', lw=0.8, alpha=0.6)
            pstr = f'p={p:.1e}' if p < 0.001 else f'p={p:.3f}'
            ax.text(0.95, 0.95, f'r={r:.2f}, {pstr}\nn={mask.sum()}',
                    transform=ax.transAxes, fontsize=7, ha='right', va='top')
        if v == 0:
            ax.set_title(f'Visible within {rlbl}', fontsize=9)
        if j == 0:
            ax.set_ylabel(f'{vname}\nMAE [{UNITS[v]}]', fontsize=7)
        if v == len(VARS) - 1:
            ax.set_xlabel('# visible neighbours', fontsize=8)

common.save_fig(fig, '49_mae_vs_vis_neighbours')
plt.show()
plt.close(fig)


In [ ]:
# ── MAE vs elevation difference to nearest visible station ───────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
fig.suptitle('Reconstruction MAE at Δ=0 vs |Δh| to nearest visible station\n'
             '(masked stations, v27 MR=0.5)', fontsize=13)

for v, (ax, vname) in enumerate(zip(axes.flat, VARS)):
    y = msk_t0[MASKED_IDX, v]
    x = elev_diff_nn
    mask = np.isfinite(y)
    ax.scatter(x[mask], y[mask], s=18, alpha=0.5, color='#7B68AE', edgecolor='none')
    if mask.sum() > 5:
        slope, intercept, r, p, _ = stats.linregress(x[mask], y[mask])
        xfit = np.linspace(x[mask].min(), x[mask].max(), 50)
        ax.plot(xfit, slope*xfit+intercept, 'k--', lw=1)
        pstr = f'p={p:.1e}' if p < 0.001 else f'p={p:.3f}'
        ax.text(0.95, 0.95, f'r={r:.2f}, {pstr}\nn={mask.sum()}',
                transform=ax.transAxes, fontsize=7, ha='right', va='top')
    ax.set_title(f'{vname} [{UNITS[v]}]', fontsize=10)
    ax.set_xlabel('|Δh| to nearest visible station [m]', fontsize=8)
    ax.set_ylabel('MAE', fontsize=8)

common.save_fig(fig, '49_mae_vs_elev_diff_vis')
plt.show()
plt.close(fig)


In [ ]:
# ── Terrain class boxplots: only masked stations ─────────────────────────────
tc_names = list(tc_idx_msk.keys())
colors_tc = ['#5BA55B', '#D35F5F', '#7BAFD4', '#E8A838']

fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
fig.suptitle('Reconstruction MAE at Δ=0 by terrain class\n'
             '(masked stations only, v27 MR=0.5)', fontsize=13)

for v, (ax, vname) in enumerate(zip(axes.flat, VARS)):
    data, labels = [], []
    for ci, tc_name in enumerate(tc_names):
        # Intersect terrain class with masked set
        idx_in_class = tc_idx_msk[tc_name]
        idx_masked_in_class = np.intersect1d(idx_in_class, MASKED_IDX)
        vals = msk_t0[idx_masked_in_class, v]
        vals = vals[np.isfinite(vals)]
        data.append(vals)
        labels.append(f'{tc_name}\n(n={len(vals)})')
    bp = ax.boxplot(data, labels=labels, patch_artist=True, showfliers=False)
    for patch, c in zip(bp['boxes'], colors_tc):
        patch.set_facecolor(c); patch.set_alpha(0.6)
    ax.set_title(f'{vname} [{UNITS[v]}]', fontsize=10)
    ax.tick_params(axis='x', rotation=25, labelsize=7)
    ax.set_ylabel('MAE', fontsize=8)

common.save_fig(fig, '49_terrain_class_masked_only')
plt.show()
plt.close(fig)


In [ ]:
# ── Map: fixed mask layout ──────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
draw_dem(ax, dem_bg, dem_fg, switzerland)
ax.scatter(stn.easting.values[VISIBLE_IDX], stn.northing.values[VISIBLE_IDX],
           s=40, c='#5BA55B', edgecolors='k', linewidths=0.3,
           zorder=5, label=f'visible (n={len(VISIBLE_IDX)})')
ax.scatter(stn.easting.values[MASKED_IDX], stn.northing.values[MASKED_IDX],
           s=40, c='#D35F5F', edgecolors='k', linewidths=0.3,
           zorder=5, label=f'masked (n={len(MASKED_IDX)})')
ax.legend(fontsize=9, loc='lower left')
ax.set_title('Fixed evaluation mask (seed=42, MR=0.5)', fontsize=12)
plt.tight_layout()
common.save_fig(fig, '49_fixed_mask_map')
plt.show()
plt.close(fig)


In [ ]:
# ── Correlation summary table ───────────────────────────────────────────────
predictors = [
    ('Dist. to nearest visible [km]', nn_vis_dist_km),
    ('# visible within 10 km', n_vis_10km),
    ('# visible within 25 km', n_vis_25km),
    ('# visible within 50 km', n_vis_50km),
    ('|Δh| to nearest visible [m]', elev_diff_nn),
    ('Elevation [m]', h_masked),
]
rows = []
for plbl, px in predictors:
    for v, vname in enumerate(VARS):
        y = msk_t0[MASKED_IDX, v]
        valid = np.isfinite(y)
        if valid.sum() < 5:
            continue
        r, p = stats.pearsonr(px[valid], y[valid])
        rows.append({'predictor': plbl, 'variable': vname,
                     'r': r, 'p': p, 'n': int(valid.sum())})
corr_df = pd.DataFrame(rows)
corr_piv = corr_df.pivot(index='predictor', columns='variable', values='r')
corr_piv = corr_piv[VARS]  # enforce column order
print('Pearson r between predictor and reconstruction MAE at Δ=0 (masked stations):')
display(corr_piv.round(3))
common.save_table(corr_df, '49_vis_metric_correlations')


## Summary

**Key findings from inpainting analysis ($\Delta=0$, v27, MR=0.5):**

1. Masked stations have 7–26× higher MAE at $t=0$ than visible ones,
   confirming that reconstruction is genuinely spatial interpolation, not
   a trivial copy from the input.
2. **Distance to nearest visible station** and **number of visible neighbours
   within 10–50 km** are the strongest spatial predictors of reconstruction
   quality — more informative than all-neighbour counts, because only
   visible stations contribute information to the decoder.
3. **Elevation difference** to the nearest visible station modulates error:
   masked stations at very different altitudes from their nearest visible
   peer are harder to reconstruct, consistent with altitude-dependent
   meteorological gradients.
4. Terrain class effects are present but confounded with isolation —
   exposed ridges and summits tend to be both topographically complex
   and spatially isolated.
5. As lead time increases, masked and visible MAE converge — the
   information advantage of having been visible at $t=0$ decays.

**Controlled progressive masking** (item 4 in the analysis plan) is not
practical without re-running inference: the fixed mask is baked into the
prediction files, so selectively masking additional surrounding stations
would require generating new predictions with modified masks. This is left
as potential future work.
